## Prepare Tiny Shakespeare Text

In [ ]:
import requests

tiny_shakespeare = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(tiny_shakespeare)
with open("shakespeare.txt", "wb") as f:
    f.write(response.content)

In [ ]:
with open("shakespeare.txt", "r") as f:
    text = f.read()

print(f"Character count: {len(set(text))}")
print(f"Text length: {len(text)}")

Character count: 65
Text length: 1115394


In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("".join(chars))

stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}


def encode(s: str):
    return [stoi[c] for c in s]


def decode(idxs: list[int]):
    return [itos[i] for i in idxs]


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [ ]:
# hyperparameters
batch_size = 32
block_size = 8
max_iters = 3000
eval_interval = 300
learning_rate = 1e-2
device = "mps" if torch.mps.is_available() else "cpu"
eval_iters = 200

## Create Dataset and DataLoaders

In [95]:
# encode entire text
import torch

data = torch.tensor(encode(text))

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

x = train_data[:block_size]
y = train_data[1 : block_size + 1]
for t in range(block_size):
    context = x[: t + 1]
    target = y[t]
    print(f"Target for input: {context} is {target}")

Target for input: tensor([18]) is 47
Target for input: tensor([18, 47]) is 56
Target for input: tensor([18, 47, 56]) is 57
Target for input: tensor([18, 47, 56, 57]) is 58
Target for input: tensor([18, 47, 56, 57, 58]) is 1
Target for input: tensor([18, 47, 56, 57, 58,  1]) is 15
Target for input: tensor([18, 47, 56, 57, 58,  1, 15]) is 47
Target for input: tensor([18, 47, 56, 57, 58,  1, 15, 47]) is 58


In [ ]:
def get_batch(split):
    # generate small batch of data of inputs x and target y
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i : i + block_size] for i in ix])
    y = torch.stack([data[i + 1 : i + block_size + 1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y


xb, yb = get_batch("train")
for x, y in zip(xb, yb):
    print(f"Targets for {x} are {y}")

Targets for tensor([43, 50, 47, 49, 43,  1, 46, 43], device='mps:0') are tensor([50, 47, 49, 43,  1, 46, 43,  1], device='mps:0')
Targets for tensor([46, 39, 58,  6,  1, 40, 63,  1], device='mps:0') are tensor([39, 58,  6,  1, 40, 63,  1, 39], device='mps:0')
Targets for tensor([42,  6,  1, 47, 52,  1, 45, 53], device='mps:0') are tensor([ 6,  1, 47, 52,  1, 45, 53, 53], device='mps:0')
Targets for tensor([33, 31, 10,  0, 37, 53, 59, 56], device='mps:0') are tensor([31, 10,  0, 37, 53, 59, 56,  1], device='mps:0')
Targets for tensor([53, 56, 39, 52, 41, 43, 11,  1], device='mps:0') are tensor([56, 39, 52, 41, 43, 11,  1, 47], device='mps:0')
Targets for tensor([56, 40, 47, 42,  2,  1, 35, 46], device='mps:0') are tensor([40, 47, 42,  2,  1, 35, 46, 43], device='mps:0')
Targets for tensor([50, 47, 43, 45, 43,  8,  0,  0], device='mps:0') are tensor([47, 43, 45, 43,  8,  0,  0, 23], device='mps:0')
Targets for tensor([47, 56,  1, 39, 40, 59, 57, 43], device='mps:0') are tensor([56,  1, 3

In [ ]:
from torch.utils.data import Dataset, DataLoader


# TODO: fill in
class ShakespeareText(Dataset):
    def __init__(self, text: str):
        super().__init__()
        self.text = text

    def __getitem__(self, index):
        pass

    def __len__(self):
        return len(self.text)


shakespeare_text = ShakespeareText(text)
train_dataloader = DataLoader(shakespeare_text, 4, shuffle=True)
val_dataloader = DataLoader()

## Bigram Language Model

In [78]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)


class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()

        # each token reacs logits for text token from lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx)  # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(
                B * T
            )  # -1 if we want pytorch to guess what value should be
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in current context
        for _ in range(max_new_tokens):
            # get predictions
            logits, _ = self(idx)

            # focus on last time step
            logits = logits[:, -1, :]

            # get probabilities
            probs = F.softmax(logits, dim=-1)  # (B, C)

            # sample from created distribution
            next_idx = torch.multinomial(probs, num_samples=1)  # (B, 1)

            # append sampled index to running sequence
            idx = torch.cat((idx, next_idx), dim=1)  # (B, T + 1)

        return idx

In [98]:
torch.manual_seed(1337)
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

In [ ]:
@torch.inference_mode()
def estimate_loss():
    out = {}
    model.eval()

    for split in ["train", "val"]:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, y = get_batch(split)
            logits, loss = model(X, y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

In [ ]:
import matplotlib.pyplot as plt

model = model.to(device)
for step in range(max_iters):
    if step % eval_interval == 0:
        losses = estimate_loss()
        print(
            f"Step {step}: train loss {losses['train']:.3f}, val loss {losses['val']:.3f}"
        )

    # sample batch
    xb, yb = get_batch("train")

    # evaluate loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

Step 0: train loss 4.728, val loss 4.726
Step 300: train loss 4.728, val loss 4.721
Step 600: train loss 4.721, val loss 4.725
Step 900: train loss 4.728, val loss 4.726
Step 1200: train loss 4.730, val loss 4.726
Step 1500: train loss 4.725, val loss 4.722
Step 1800: train loss 4.724, val loss 4.725
Step 2100: train loss 4.722, val loss 4.721
Step 2400: train loss 4.726, val loss 4.727
Step 2700: train loss 4.722, val loss 4.725


In [76]:
@torch.inference_mode()
def generate_text(legnth: int):
    print(
        "".join(
            decode(
                model.generate(
                    idx=torch.zeros((1, 1), dtype=torch.long), max_new_tokens=legnth
                )[0].tolist()
            )
        )
    )

In [93]:
model.to("cpu")
generate_text(500)


W:zYno qe$srJ$IDUPKDktw.szP q-gOcUipHZYOfa!uRlgHASYlgChF$zLNoiviAF.?RvRSV-oipkzC,rDW3czkzdaY
Uq.RjKu?R:VByFqxq-Qbd
JGbOda!wkgYhVkpJw!?hKoiF$OlyDoRMwbiOdCOdc&uEo&ulta!xTrgZBju,ehmJ?EJMX3KXEC CjTzChXj$HVbowhz;;A:aXHoItxlgOvHeCjuDTi.cLqKDnbVehs!uCKhslueSxaIBXNvZ!AKO:,ivulx:Mlf-!?xaG:.tt-!BjSJJsrnp.lviOIIQNmJtDnQERSPA: DlI,U,F-Z;OJ zkUvVktkQbb
otR-gCbj,SpJAXfZv!wWPyIyNx?SZ-tjxlcYGJU-,SKzoBi-vjxJFdIN!pwN3C-LSrP,jLU;pwa!AUhZvLtRlap&!AHB&vHW
KkOjxj nYw&ylykzkduQjbvv$ flxjaqIrDMlvv&!NQeuvjpJgg
3fNDGgZdz


In [ ]:
# layer norm
class BatchNorm1d:
    def __init__(self, dim, eps=1e-5, momentum=0.1) -> None:
        self.eps = eps
        self.momentum = momentum
        self.training = True

        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x: torch.Tensor) -> Any:
        # calculate forward pass
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)